# Logprob Metric Distributions

Visualize scalar logprob metrics and token similarity features. Notebook cells set experiment parameters and call version-controlled helpers from `logprob_metrics_distributions_utils.py`.

In [ ]:
from __future__ import annotations

from IPython.display import display

import pandas as pd
import seaborn as sns

from notebooks.utils.logprob_metrics_distributions_utils import (
    DEFAULT_METRICS,
    available_metric_columns,
    filter_metrics,
    load_metrics,
    load_token_similarity_stats,
    metric_summary,
    plot_aggregate_similarity_scatter,
    plot_experiment_metric_row,
    plot_global_similarity_scatter,
    plot_hg_metric_mean_std_kde_by_model,
    plot_metric_distributions,
    plot_split_metric_comparison,
    plot_similarity_scatter_pair,
    split_metric_summary,
    plot_token_similarity,
)
from scripts.data_root import resolve_data_root

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 100)

DATA_ROOT = resolve_data_root()
PROCESSED_ROOT = DATA_ROOT / "processed"

## Load Metrics

In [ ]:
metrics_df = load_metrics(PROCESSED_ROOT)
metrics_df.head()

In [ ]:
metric_summary(metrics_df)

## Configure Filters

In [ ]:
selected_datasets = None
selected_models = None
selected_metrics = [metric for metric in DEFAULT_METRICS if metric in available_metric_columns(metrics_df)]

plot_df = filter_metrics(metrics_df, selected_datasets, selected_models)

print(f"Rows selected: {len(plot_df):,}")
print(f"Datasets: {sorted(plot_df['dataset'].unique())}")
print(f"Models: {sorted(plot_df['model'].unique())}")
print(f"Metrics: {selected_metrics}")

## Metric Distributions

In [ ]:
for metric in selected_metrics:
    plot_metric_distributions(plot_df, metric)

In [ ]:
for dataset, model in plot_df[["dataset", "model"]].drop_duplicates().itertuples(index=False):
    plot_experiment_metric_row(plot_df, dataset, model, selected_metrics)

## Numeric Summary

In [ ]:
plot_df.groupby(["dataset", "model", "split"])[selected_metrics].describe().round(4)

## HG/MG D-Score Comparison

`d_score_top20` is a passage-level standardized contrast, not a token probability. `d_score_top20_coverage` is the mean probability mass in the available top-20 candidate list. The following summary and plots provide descriptive HG/MG comparisons by dataset and model; they do not train or evaluate a classifier.

In [ ]:
d_score_metrics = [
    metric
    for metric in ("d_score_top20", "d_score_top20_coverage")
    if metric in selected_metrics
]
if d_score_metrics:
    display(split_metric_summary(plot_df, d_score_metrics))
    plot_split_metric_comparison(plot_df, d_score_metrics)
else:
    print("D-score columns are not available in the loaded metrics.")

## Token Similarity Scatter

In [ ]:
similarity_stats_df = load_token_similarity_stats(PROCESSED_ROOT)
similarity_stats_df.head()

In [ ]:
plot_hg_metric_mean_std_kde_by_model(
    metrics_df,
    similarity_stats_df,
)

In [ ]:
plot_hg_metric_mean_std_kde_by_model(
    metrics_df,
    similarity_stats_df,
    split="MG",
)

In [ ]:
similarity_plot_type = "kde"  # "scatter" or "kde"
similarity_datasets = None
similarity_models = None

plot_token_similarity(
    similarity_stats_df,
    plot_type=similarity_plot_type,
    selected_datasets=similarity_datasets,
    selected_models=similarity_models,
)

## Aggregate Similarity Scatter

In [ ]:
plot_aggregate_similarity_scatter(similarity_stats_df, "model")
plot_aggregate_similarity_scatter(similarity_stats_df, "dataset")

## Global Similarity Scatter

In [ ]:
plot_global_similarity_scatter(similarity_stats_df)